In [ ]:
# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
warnings.filterwarnings('ignore')
tqdm.pandas() # Activa la barra de progreso de Pandas

# Configuramos la ruta local a tu carpeta de datos
PATH_DATA = 'data'
if not os.path.exists(PATH_DATA):
    print(f" Error: No se encuentra la carpeta '{PATH_DATA}'. Asegúrate de que existe.")

# Verificamos si hay GPU disponible en tu equipo local
if torch.cuda.is_available():
    device = 0
    print(" Procesando con: GPU (NVIDIA)")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 0
    print(" Procesando con: GPU ")
else:
    device = -1
    print(" Procesando con: CPU ")

In [ ]:
# ==============================================================================
# 2. CARGA Y MUESTREO ESTRATIFICADO (TOP 10 POR PISO)
# ==============================================================================
ruta_reviews = os.path.join(PATH_DATA, 'reviews.csv')

# Cargamos solo las columnas necesarias para no saturar la RAM
try:
    df_reviews = pd.read_csv(ruta_reviews, usecols=['listing_id', 'comments', 'date']).dropna()
    df_reviews = df_reviews.sort_values(by=['listing_id', 'date'], ascending=[True, False])
except Exception:
    print(" No se encontró la columna 'date', cargando sin orden temporal...")
    df_reviews = pd.read_csv(ruta_reviews, usecols=['listing_id', 'comments']).dropna()

print(" Aplicando muestreo (Top 10 reseñas por alojamiento)...")
df_reviews = df_reviews.groupby('listing_id').head(10).copy()

def limpiar_texto(texto):
    texto = str(texto)
    texto = re.sub(r'<br\s*/?>|[\r\n]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

df_reviews['comments'] = df_reviews['comments'].apply(limpiar_texto)
# Filtramos comentarios que se queden muy cortos tras limpiar
df_reviews = df_reviews[df_reviews['comments'].str.len() > 10].copy()

print(f" Listas para procesar: {len(df_reviews)} reseñas.")


In [ ]:

# ==============================================================================
# 3. INFERENCIA CON TRANSFORMER MULTILINGÜE
# ==============================================================================

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=512,
    device=device
)

def obtener_score(texto):
    try:
        resultado = sentiment_analyzer(texto[:512])[0]
        # Transforma de '1-5 stars' a un score de 0.0 a 1.0
        return (int(resultado['label'].split()[0]) - 1) / 4.0
    except:
        return 0.5 # Nota neutra si hay error

# progress_apply muestra una barra de carga para que sepas por dónde va
df_reviews['score_sentimiento'] = df_reviews['comments'].progress_apply(obtener_score)



In [ ]:
# ==============================================================================
# 4. AGRUPACIÓN Y GUARDADO FINAL
# ==============================================================================
df_nlp_final = df_reviews.groupby('listing_id')['score_sentimiento'].mean().reset_index()

ruta_salida = os.path.join(PATH_DATA, 'reviews_scores_nlp.csv')
df_nlp_final.to_csv(ruta_salida, index=False)

print(f" El archivo final está guardado en: {ruta_salida}")